In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
from ipywidgets import interactive, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, HTMLMath
from IPython.display import display

# ------------------------------------------------------------
# CONTROLS
# ------------------------------------------------------------

filter_title = HTML(value="<b>Filter Type:</b>")
filter_radio = RadioButtons(options=['Low-pass', 'High-pass'], value='Low-pass', description='', layout=Layout(width='150px'))

rc_title = HTML(value="<b>RC Filter</b>")
rl_title = HTML(value="<b>RL Filter</b>")

R_RC = FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0, description='R_RC (kΩ):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))
C_RC = FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0, description='C (μF):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))

R_RL = FloatSlider(min=0.1, max=10.0, step=0.1, value=2.0, description='R_RL (kΩ):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))
L_RL = FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0, description='L (H):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))

# ------------------------------------------------------------
# CONTROL LAYOUT
# ------------------------------------------------------------

filter_box = VBox([filter_title, filter_radio], layout=Layout(width='180px'))

rc_box = VBox([rc_title, R_RC, C_RC], layout=Layout(width='340px'))
rl_box = VBox([rl_title, R_RL, L_RL], layout=Layout(width='340px'))

control_row = HBox([rc_box, rl_box], layout=Layout(width='720px', justify_content='space-between', align_items='flex-start'))

# ------------------------------------------------------------
# NUMERICAL OUTPUT
# ------------------------------------------------------------

parameter_title = HTML(value="<b>Calculated Filter Parameters</b>")

tau_output = HTMLMath(layout=Layout(width='800px'))
omega_output = HTMLMath(layout=Layout(width='800px'))
frequency_output = HTMLMath(layout=Layout(width='800px'))
error_output = HTMLMath(layout=Layout(width='800px'))
status_output = HTML(layout=Layout(width='800px'))

parameter_box = VBox([parameter_title, tau_output, omega_output, frequency_output, error_output, status_output], layout=Layout(width='820px'))

# ------------------------------------------------------------
# RESPONSE FUNCTION
# ------------------------------------------------------------

def calculate_response(filter_type, omega, tau_RC, tau_RL):

    jw = 1j * omega

    if filter_type == 'Low-pass':
        H_RC = 1.0 / (1.0 + jw * tau_RC)
        H_RL = 1.0 / (1.0 + jw * tau_RL)

    else:
        H_RC = jw * tau_RC / (1.0 + jw * tau_RC)
        H_RL = jw * tau_RL / (1.0 + jw * tau_RL)

    magnitude_RC = 20.0 * np.log10(np.maximum(np.abs(H_RC), 1e-12))
    magnitude_RL = 20.0 * np.log10(np.maximum(np.abs(H_RL), 1e-12))

    phase_RC = np.degrees(np.angle(H_RC))
    phase_RL = np.degrees(np.angle(H_RL))

    return magnitude_RC, magnitude_RL, phase_RC, phase_RL

# ------------------------------------------------------------
# MAIN INTERACTIVE FUNCTION
# ------------------------------------------------------------

def plot_rc_rl_filter(filter_type, R_RC_kohm, C_RC_uf, R_RL_kohm, L_RL_h):

    Rrc = R_RC_kohm * 1e3
    C = C_RC_uf * 1e-6

    Rrl = R_RL_kohm * 1e3
    L = L_RL_h

    tau_RC = Rrc * C
    tau_RL = L / Rrl

    wc_RC = 1.0 / tau_RC
    wc_RL = 1.0 / tau_RL

    fc_RC = wc_RC / (2.0 * np.pi)
    fc_RL = wc_RL / (2.0 * np.pi)

    equivalence_error = 100.0 * abs(tau_RC - tau_RL) / max(tau_RC, tau_RL)

    # --------------------------------------------------------
    # CALCULATED PARAMETERS
    # --------------------------------------------------------

    tau_output.value = rf"$$\tau_{{RC}}=R_{{RC}}C={tau_RC:.6f}\ \mathrm{{s}}\qquad\qquad\tau_{{RL}}=\frac{{L}}{{R_{{RL}}}}={tau_RL:.6f}\ \mathrm{{s}}$$"

    omega_output.value = rf"$$\omega_{{c,RC}}=\frac{{1}}{{\tau_{{RC}}}}={wc_RC:.2f}\ \mathrm{{rad/s}}\qquad\qquad\omega_{{c,RL}}=\frac{{1}}{{\tau_{{RL}}}}={wc_RL:.2f}\ \mathrm{{rad/s}}$$"

    frequency_output.value = rf"$$f_{{c,RC}}={fc_RC:.2f}\ \mathrm{{Hz}}\qquad\qquad f_{{c,RL}}={fc_RL:.2f}\ \mathrm{{Hz}}$$"

    error_output.value = rf"$$\mathrm{{Equivalence\ Error}}=100\frac{{|\tau_{{RC}}-\tau_{{RL}}|}}{{\max(\tau_{{RC}},\tau_{{RL}})}}={equivalence_error:.2f}\%$$"

    if np.isclose(tau_RC, tau_RL, rtol=0.0, atol=1e-12):
        status_output.value = "<div style='font-size:15px;'><b>RC and RL filters are equivalent: τ<sub>RC</sub> = τ<sub>RL</sub></b></div>"

    else:
        status_output.value = "<div style='font-size:15px;'><b>Adjust the circuit parameters until the Equivalence Error becomes 0%.</b></div>"

    # --------------------------------------------------------
    # FREQUENCY RANGE
    # ------------------------------------------------------------

    wc_min = min(wc_RC, wc_RL)
    wc_max = max(wc_RC, wc_RL)

    omega = np.logspace(np.log10(wc_min / 100.0), np.log10(wc_max * 100.0), 1200)

    # --------------------------------------------------------
    # FREQUENCY RESPONSES
    # ------------------------------------------------------------

    mag_RC, mag_RL, phase_RC, phase_RL = calculate_response(filter_type, omega, tau_RC, tau_RL)

    # --------------------------------------------------------
    # NEW FIGURE FOR EACH UPDATE
    # ------------------------------------------------------------

    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))

    ax_mag = axes[0]
    ax_phase = axes[1]

    # --------------------------------------------------------
    # MAGNITUDE RESPONSE
    # ------------------------------------------------------------

    ax_mag.semilogx(omega, mag_RC, linewidth=2.0, label='RC filter')
    ax_mag.semilogx(omega, mag_RL, '--', linewidth=2.0, label='RL filter')

    ax_mag.axvline(wc_RC, linestyle=':', linewidth=1.2)
    ax_mag.axvline(wc_RL, linestyle=':', linewidth=1.2)

    ax_mag.axhline(-3.0103, linestyle=':', linewidth=1.0)

    ax_mag.set_title('Magnitude Response')
    ax_mag.set_xlabel('Angular Frequency ω (rad/s)')
    ax_mag.set_ylabel('Magnitude (dB)')
    ax_mag.set_xlim(wc_min / 100.0, wc_max * 100.0)
    ax_mag.set_ylim(-60, 5)
    ax_mag.grid(True, which='both', linestyle=':', alpha=0.7)
    ax_mag.legend(loc='best')

    # --------------------------------------------------------
    # PHASE RESPONSE
    # ------------------------------------------------------------

    ax_phase.semilogx(omega, phase_RC, linewidth=2.0, label='RC filter')
    ax_phase.semilogx(omega, phase_RL, '--', linewidth=2.0, label='RL filter')

    ax_phase.axvline(wc_RC, linestyle=':', linewidth=1.2)
    ax_phase.axvline(wc_RL, linestyle=':', linewidth=1.2)

    if filter_type == 'Low-pass':
        phase_reference = -45.0
        phase_min = -100
        phase_max = 10

    else:
        phase_reference = 45.0
        phase_min = -10
        phase_max = 100

    ax_phase.axhline(phase_reference, linestyle=':', linewidth=1.0)

    ax_phase.set_title('Phase Response')
    ax_phase.set_xlabel('Angular Frequency ω (rad/s)')
    ax_phase.set_ylabel('Phase (deg)')
    ax_phase.set_xlim(wc_min / 100.0, wc_max * 100.0)
    ax_phase.set_ylim(phase_min, phase_max)
    ax_phase.grid(True, which='both', linestyle=':', alpha=0.7)
    ax_phase.legend(loc='best')

    # --------------------------------------------------------
    # AXIS FORMAT
    # ------------------------------------------------------------

    for ax in axes:

        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.tick_params(axis='x', labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
        ax.set_frame_on(True)

        for spine in ['left', 'right', 'top', 'bottom']:

            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_linewidth(0.8)
            ax.spines[spine].set_clip_on(False)

    fig.subplots_adjust(left=0.08, right=0.97, bottom=0.16, top=0.88, wspace=0.28)

    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# INTERACTIVE WIDGET
# ------------------------------------------------------------

widget_plot = interactive(plot_rc_rl_filter, filter_type=filter_radio, R_RC_kohm=R_RC, C_RC_uf=C_RC, R_RL_kohm=R_RL, L_RL_h=L_RL)

# ------------------------------------------------------------
# OUTPUT OF INTERACTIVE
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

# ------------------------------------------------------------
# REMOVE OUTPUT SCROLL BARS
# ------------------------------------------------------------

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.output_scroll {
    max-height: none !important;
    height: auto !important;
    overflow: visible !important;
    overflow-y: visible !important;
    overflow-x: visible !important;
}

</style>
"""))

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display(HTML("<h3>First-Order RC–RL Filter Comparison</h3>"))
display(filter_box)
display(control_row)
display(parameter_box)
display(plot_output)